<a href="https://colab.research.google.com/github/Ndeathn0te/Scan-to-BIM-Dynamo/blob/master/notebooks/colab-github-demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Using Google Colab with GitHub



In [ ]:
# Pegar entero en una celda de Colab (entorno GPU T4) y ejecutar. Ver INSTRUCCIONES al principio de colab_hvac_les.ipynb
import time
T_INICIO = time.time()
HORAS = 11.5        # Colab gratuito corta hacia las 12 h (a veces antes)
# Casos que no se calculan aquí: la sensibilidad 0,20 m (no cabe en una sesión de Colab) y los ya descargados
SALTAR = "este_n3_098_v3_dx020"
SOLO = ""           # vacío = la cola completa, desde el final (la RTX 3050 la hace desde el principio)
PRUEBA = False      # True: solo voxeliza un caso (minutos) para comprobar el paquete
SALIDA = '/content/resultados'

import glob, os, shutil, subprocess, sys, zipfile
from google.colab import files
zips = glob.glob('/content/hvac_nube*.zip')
if not zips:
    up = files.upload()
    zips = [os.path.join('/content', n) for n in up if n.startswith('hvac_nube') and n.endswith('.zip')]
assert zips, 'Sube hvac_nube.zip'
shutil.rmtree('/content/hvac', ignore_errors=True)
print('paquete:', zips[-1]); zipfile.ZipFile(zips[-1]).extractall('/content/hvac')
PRUEBA_CUPY = ("import cupy as cp, cupyx.scipy.fft as f; a = cp.ones((64, 64, 64), cp.float32); f.dctn(a); "
               "print('CuPy', cp.__version__, 'OK', cp.cuda.runtime.getDeviceCount(), 'GPU', "
               "cp.cuda.runtime.getDeviceProperties(0)['name'].decode())")
def cupy_ok():
    r = subprocess.run([sys.executable, '-c', PRUEBA_CUPY], capture_output=True, text=True)
    print((r.stdout + r.stderr).strip()[-400:]); return r.returncode == 0
if not PRUEBA and not cupy_ok():
    for paq in ('cupy-cuda12x', 'cupy-cuda12x[ctk]'):
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', paq])
        if cupy_ok(): break
    else:
        raise SystemExit('CuPy no funciona: el entorno de ejecución tiene que ser GPU T4')
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

import types
os.makedirs(SALIDA, exist_ok=True)
sys.path[:0] = ['/content/hvac/scripts', '/content/hvac/scripts/nube']; os.chdir('/content/hvac')
import kaggle_cola
pend = [c['tag'] for c in kaggle_cola.cola(types.SimpleNamespace(saltar=SALTAR, solo=SOLO, salida=SALIDA))]
print('cola de Colab:', ', '.join(pend))
for tag in pend:
    horas = HORAS - (time.time() - T_INICIO) / 3600 - 0.1
    if horas < 0.6: print('sin tiempo para otro caso'); break
    args = [sys.executable, '-u', 'scripts/nube/kaggle_cola.py', '--salida', SALIDA, '--horas', f'{horas:.2f}',
            '--ngpu', '1', '--solo', tag] + (['--prueba'] if PRUEBA else [])
    p = subprocess.Popen(args, cwd='/content/hvac', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for linea in p.stdout:
        print(linea, end='', flush=True)
    p.wait(); print('código de salida', p.returncode, flush=True)
    hechos = [f for f in glob.glob(SALIDA + '/**/*', recursive=True) if os.path.isfile(f) and tag in os.path.basename(f)]
    if not any(f.endswith(f'roof_{tag}.json') for f in hechos):
        print('no terminó', tag, '(¿no cabía en el tiempo?): paro la cola'); break
    z = f'/content/resultados_colab_{tag}.zip'
    with zipfile.ZipFile(z, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in hechos + [os.path.join(SALIDA, 'ESTADO.txt')]:
            if os.path.exists(f): zf.write(f, os.path.relpath(f, SALIDA))
    files.download(z); print('descargado', os.path.basename(z), flush=True)
    if PRUEBA: break


Saving hvac_nube.zip to hvac_nube.zip
paquete: /content/hvac_nube.zip
CuPy 14.0.1 OK 1 GPU Tesla T4
Wed Nov 25 14:12:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P0             30W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                       


[Google Colaboratory](http://colab.research.google.com) is designed to integrate cleanly with GitHub, allowing both loading notebooks from github and saving notebooks to github.

## Loading Public Notebooks Directly from GitHub

Colab can load public github notebooks directly, with no required authorization step.

For example, consider the notebook at this address: https://github.com/googlecolab/colabtools/blob/main/notebooks/colab-github-demo.ipynb.

The direct colab link to this notebook is: https://colab.research.google.com/github/googlecolab/colabtools/blob/main/notebooks/colab-github-demo.ipynb.

To generate such links in one click, you can use the [Open in Colab](https://chrome.google.com/webstore/detail/open-in-colab/iogfkhleblhcpcekbiedikdehleodpjo) Chrome extension.

## Browsing GitHub Repositories from Colab

Colab also supports special URLs that link directly to a GitHub browser for any user/organization, repository, or branch. For example:

- http://colab.research.google.com/github will give you a general github browser, where you can search for any github organization or username.
- http://colab.research.google.com/github/googlecolab/ will open the repository browser for the ``googlecolab`` organization. Replace ``googlecolab`` with any other github org or user to see their repositories.
- http://colab.research.google.com/github/googlecolab/colabtools/ will let you browse the main branch of the ``colabtools`` repository within the ``googlecolab`` organization. Substitute any user/org and repository to see its contents.
- http://colab.research.google.com/github/googlecolab/colabtools/blob/main will let you browse ``main`` branch of the ``colabtools`` repository within the ``googlecolab`` organization. (don't forget the ``blob`` here!) You can specify any valid branch for any valid repository.

## Loading Private Notebooks

Loading a notebook from a private GitHub repository is possible, but requires an additional step to allow Colab to access your files.
Do the following:

1. Navigate to http://colab.research.google.com/github.
2. Click the "Include Private Repos" checkbox.
3. In the popup window, sign-in to your Github account and authorize Colab to read the private files.
4. Your private repositories and notebooks will now be available via the github navigation pane.

## Saving Notebooks To GitHub or Drive

Any time you open a GitHub hosted notebook in Colab, it opens a new editable view of the notebook. You can run and modify the notebook without worrying about overwriting the source.

If you would like to save your changes from within Colab, you can use the File menu to save the modified notebook either to Google Drive or back to GitHub. Choose **File→Save a copy in Drive** or **File→Save a copy to GitHub** and follow the resulting prompts. To save a Colab notebook to GitHub requires giving Colab permission to push the commit to your repository.

## Open In Colab Badge

Anybody can open a copy of any github-hosted notebook within Colab. To make it easier to give people access to live views of GitHub-hosted notebooks,
colab provides a [shields.io](http://shields.io/)-style badge, which appears as follows:

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/googlecolab/colabtools/blob/main/notebooks/colab-github-demo.ipynb)

The markdown for the above badge is the following:

```markdown
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/googlecolab/colabtools/blob/main/notebooks/colab-github-demo.ipynb)
```

The HTML equivalent is:

```HTML
<a href="https://colab.research.google.com/github/googlecolab/colabtools/blob/main/notebooks/colab-github-demo.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>
```

Remember to replace the notebook URL in this template with the notebook you want to link to.